In [11]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pickle
from pyproj import Geod
from scipy.optimize import root_scalar
from bluemath_tk.tcs.tracks import track_triming_circle
from numpy import polyfit

geod = Geod(ellps='WGS84')

from utils.tcs_fcts import point_from_center, compute_entrance_exit_azimuth_prolong, mean_translation_velocity, furthest_point_from_entry_exit, travel_time_linear_v, solve_velocity, interpolate_geodesic, integrate_distance


* Data access can be requested to bluemath@unican.es
You can update any path or add new paths with the update_paths function, from bluemath_tk.config.paths.
Example: update_paths({'SHYTCWAVES_COEFS': '/new/path/to/data'})


In [12]:
lon_site = -75.5730
lat_site = 36.0050
RADIUS_DEG = 7
RADIUS_KM = 777

In [13]:
def extract_synthetic_track_params(tracks_ds, lat_site, lon_site, RADIUS_DEG, out_path, fig_format='figures/13_syntetic_params_new/emu_tracks_ec_earth3_veg_lr_historical_corrected/storm_{i}_comparison.png'):

    params_list = []
    df_full_list, df_small_list = [], []
    n_small = 0
    n_short = 0

    for i in range(len(tracks_ds['storm_number'])):

        storm = tracks_ds.isel(storm_number=i)
        #name = f"{storm.name.values.item().decode('utf-8')}_{storm.time.values[0].astype('datetime64[Y]').astype(int) + 1970}"

        df_full = pd.DataFrame({
            'time': storm['time'].values,
            'lat': storm['lat_trks'].values,
            'lon': ((storm['lon_trks'].values + 180) % 360) - 180,
            'pmin': storm['pmin_trks'].values,
    #        'vmax': storm['wmo_wind'].values
        }).dropna()
        #df_full_list.append(df_full.copy())
        ref_time = pd.Timestamp("2000-01-01 00:00:00")
        df_full["time"] = ref_time + pd.to_timedelta(df_full["time"], unit="s")

        df = df_full.drop_duplicates(subset='time').set_index('time')
        df = df.reindex(pd.date_range(df.index.min(), df.index.max(), freq='H')).interpolate()
        try:
            df_small = track_triming_circle(df, lon_site, lat_site, RADIUS_DEG)
            
        except:
            n_short += 1
            continue
        if len(df_small) < 10:
            n_small += 1
            continue

        df_full_list.append(df_full.copy())
        df_small_list.append(df_small.copy())

        entr_az, entr_d, exit_az, exit_d = compute_entrance_exit_azimuth_prolong(df_small, lat_site, lon_site)
        fur_idx, fur_dist, fur_az = furthest_point_from_entry_exit(df_small, lat_site, lon_site)
        fur_pos = df_small.index.get_loc(fur_idx)
        
        p_entr, p_fur, p_exit = df_small.iloc[0]['pmin'], df_small.iloc[fur_pos]['pmin'], df_small.iloc[-1]['pmin']

        lon_f, lat_f = df_small.iloc[fur_pos][['lon', 'lat']]

        d12_obs = geod.inv(df_small.iloc[0]['lon'], df_small.iloc[0]['lat'], lon_f, lat_f)[2] / 1000
        d23_obs = geod.inv(lon_f, lat_f, df_small.iloc[-1]['lon'], df_small.iloc[-1]['lat'])[2] / 1000

        v_fur = (mean_translation_velocity(df_small, fur_pos-1, fur_pos) + 
                mean_translation_velocity(df_small, fur_pos, fur_pos+1)) / 2
        t1 = (df_small.index[fur_pos] - df_small.index[0]).total_seconds() / 3600
        t2 = (df_small.index[-1] - df_small.index[fur_pos]).total_seconds() / 3600
        v_entr = solve_velocity(d12_obs, t1, v_fur)
        v_exit = solve_velocity(d23_obs, t2, v_fur)
        
        params_list.append({
        #    'name': name,
            'date_furthest': df_small.index[fur_pos],
            'azimuth_entrance': entr_az,
            'distance_entrance': entr_d,
            'azimuth_exit': exit_az,
            'distance_exit': exit_d,
            'azimuth_furthest': fur_az,
            'distance_furthest_km': fur_dist,
            'p_furthest_hPa': p_fur,
            'p_entrance_hPa': p_entr,
            'p_exit_hPa': p_exit,
            'vmean_entrance_kmh': v_entr,
            'vmean_furthest_kmh': v_fur,
            'vmean_exit_kmh': v_exit,
            'original_storm_id': storm['original_storm_id'].values,
            'time_emulation': storm['time_emulation'].values,
            'sim_number': storm['sim_number'].values
        })

    print(f"Extracted parameters for {len(params_list)} tracks.")
    print(f"Tracks with too few points:: {n_small}")
    print(f"Tracks outside {n_short}")

    df_synth_list = []

    for i, p in enumerate(params_list):
        v_entr, v_fur, v_exit = p['vmean_entrance_kmh'], p['vmean_furthest_kmh'], p['vmean_exit_kmh']
        
        # Skip invalid velocities
        if v_entr <= 0 or v_fur <= 0 or v_exit <= 0:
            df_synth_list.append(None)
            continue

        lat_f, lon_f = point_from_center(lat_site, lon_site, p['azimuth_furthest'], p['distance_furthest_km'])
        lat_e, lon_e = point_from_center(lat_site, lon_site, p['azimuth_entrance'],  p['distance_entrance'])
        lat_x, lon_x = point_from_center(lat_site, lon_site, p['azimuth_exit'], p['distance_exit'])

        d_fur_to_entry = geod.inv(lon_f, lat_f, lon_e, lat_e)[2] / 1000
        d_fur_to_exit = geod.inv(lon_f, lat_f, lon_x, lat_x)[2] / 1000

        dt1 = travel_time_linear_v(d_fur_to_entry, v_fur, v_entr)
        dt2 = travel_time_linear_v(d_fur_to_exit, v_fur, v_exit)
        
        if np.isnan(dt1) or np.isnan(dt2):
            df_synth_list.append(None)
            continue

        t1_arr, v1_arr, d1_arr = integrate_distance(v_fur, v_entr, dt1)

        t1_arr, v1_arr, d1_arr = t1_arr[::-1], v1_arr[::-1], d1_arr[-1] - d1_arr[::-1]
        lat1, lon1 = interpolate_geodesic(lat_e, lon_e, lat_f, lon_f, d1_arr)
        
        t2_arr, v2_arr, d2_arr = integrate_distance(v_fur, v_exit, dt2)
        lat2, lon2 = interpolate_geodesic(lat_f, lon_f, lat_x, lon_x, d2_arr)

        p_entr, p_fur, p_exit = p['p_entrance_hPa'], p['p_furthest_hPa'], p['p_exit_hPa']

        n1, n2 = len(t1_arr), len(t2_arr)
        date_fur = pd.to_datetime(p['date_furthest'])
        time_index = pd.date_range(
            start=date_fur - pd.Timedelta(hours=n1-1),
            periods=n1 + n2 - 1,
            freq='H'
        )
        
        df_synth = pd.DataFrame({
            'lat': np.concatenate([lat1, lat2[1:]]),
            'lon': np.concatenate([lon1, lon2[1:]]),
            'pmin': np.concatenate([np.linspace(p_entr, p_fur, n1), 
                                    np.linspace(p_fur, p_exit, n2)[1:]]),
            'vmean': np.concatenate([v1_arr[:-1], [v_fur], v2_arr[1:]])
        }, index=time_index)
        df_synth_list.append(df_synth)

        if i%100 == 0:

            fig = plt.figure(figsize=(15, 5))

            ax1 = plt.subplot(1, 3, 1, projection=ccrs.PlateCarree())
            ax1.add_feature(cfeature.COASTLINE); ax1.add_feature(cfeature.LAND, alpha=0.3)
            ax1.plot(df_full_list[i]['lon'], df_full_list[i]['lat'], 'k-', lw=1, label='Full')
            ax1.plot(df_small_list[i]['lon'], df_small_list[i]['lat'], 'r-', lw=2, label='Obs 7°')
            ax1.plot(df_synth['lon'], df_synth['lat'], 'b--', marker='o', ms=0.5, lw=0.1, label='Synth')
            angles = np.linspace(0, 2 * np.pi, 361)
            ax1.plot(lon_site + RADIUS_DEG * np.cos(angles) / np.cos(np.radians(lat_site)),
                    lat_site + RADIUS_DEG * np.sin(angles), 'k--', lw=1)
            #ax1.set_title(p['name']); ax1.legend(fontsize=7)
            
            ax2 = plt.subplot(1, 3, 2)
            ax2 = plt.subplot(1, 3, 2)
            ax2.plot(df_small_list[i].index, df_small_list[i]['pmin'].values, 'r-', lw=2, label='Obs')
            ax2.plot(df_synth.index, df_synth['pmin'].values, 'b--', lw=2, label='Synth')
            ax2.axvline(date_fur, color='green', ls=':', lw=1, label='Furthest')
            ax2.set_ylabel('Pressure (hPa)')
            ax2.legend(); ax2.grid(alpha=0.3)
            plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
            
            ax3 = plt.subplot(1, 3, 3)
            t_vel_obs = df_small_list[i].index[:-1]  # velocity is between points
            v_obs = [mean_translation_velocity(df_small_list[i], j, j+1) for j in range(len(df_small_list[i])-1)]
            ax3.plot(t_vel_obs, v_obs, 'r-', lw=2, label='Obs')
            ax3.plot(df_synth.index, df_synth['vmean'].values, 'b--', lw=2, label='Synth')
            ax3.axvline(date_fur, color='green', ls=':', lw=1, label='Furthest')
            ax3.set_ylabel('Velocity (km/h)')
            ax3.legend(); ax3.grid(alpha=0.3)
            plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')
            
            plt.tight_layout()
            plt.savefig(fig_format.format(i=i), dpi=300)
            plt.close()

    valid_synth = [(i, df) for i, df in enumerate(df_synth_list) if df is not None]
    valid_indices = [i for i, df in valid_synth]

    params_valid = [params_list[i] for i in valid_indices]
    df_params = pd.DataFrame(params_valid)
    #df_params['bmus'] = hist_tracks_nc['bmus'].values
    df_params.to_csv(out_path, index=False)
    df_params


In [14]:
def reconstruct_synthetic_tracks(csv_path, output_path, fig_path):

    df_parametros = pd.read_csv(csv_path)

    df_parametros_reconstruct = df_parametros.drop(columns=['date_furthest'])
    variables = ['Hsig', 'TPsmoo', 'Windv_x', 'Windv_y', 'Dir_x', 'Dir_y']

    ds_predicted_test_all = {}

    for var in variables:

        with open(f"outputs/pca_model_{var}_OK.pkl", "rb") as f:
            pca_output_all = pickle.load(f)

        with open(f"outputs/rbf_model_{var}_OK.pkl", "rb") as f:
            rbf_model = pickle.load(f)

        predictions = rbf_model.predict(dataset=df_parametros_reconstruct)

        ds_predicted_test = xr.Dataset(
            {"PCs": (("case_num", "n_component"), predictions.values)},
            coords={
                "case_num": df_parametros_reconstruct.index.values,
                "n_component": np.arange(predictions.shape[1]),
            },
        )

        da = list(pca_output_all.inverse_transform(ds_predicted_test).data_vars.values())[0]

        ds_predicted_test_all[var] = da


    # concatenation into dataset
    ds_predicted_test_all = xr.Dataset(ds_predicted_test_all)

    ds_predicted_test_all["Dir"] = (360 - np.arctan2(ds_predicted_test_all["Dir_x"], ds_predicted_test_all["Dir_y"]) * 180 / np.pi + 90) % 360

    ds_predicted_test_all = ds_predicted_test_all.drop_vars(["Dir_x", "Dir_y"])

    ds_predicted_test_all.to_netcdf(output_path)
    for i in range(len(ds_predicted_test_all['case_num'])):
        if i%100 == 0:

            ds_case = ds_predicted_test_all.sel(case_num = i)
            common_time = ds_case.time.values
            n_rows = len(ds_case.data_vars)
            fig, axes = plt.subplots(n_rows, 1, figsize=(10, 2 * n_rows), sharex=True)
            axes = np.atleast_2d(axes)
            variables = list(ds_case.data_vars.keys())
            time_mean = common_time.mean()

            point_names = ["Point 1", "Point 2", "Point 3", "Point 4"]

            for k, var in enumerate(variables):
                ax = axes.flat[k]
                arr = ds_case[var].values
                for j in range(arr.shape[1]):
                    ax.plot(common_time, arr[:, j], label=point_names[j])
                ax.axvline(time_mean, color="k", ls="--", lw=1, alpha=0.7)
                ax.set_ylabel(var)
                if k == 0:
                    ax.legend(loc="best", fontsize=8)
                ax.grid(True, alpha=0.3)
            plt.savefig(fig_path.format(i=i), dpi=300)
            plt.close()
    

In [24]:
def add_field_2_dataset(ds, csv_path, fealds):
    df_parametros = pd.read_csv(csv_path)
    for field_name in fealds:
        data_array = xr.DataArray(
            df_parametros[field_name].values,
            coords={"case_num": df_parametros.index.values},
            dims=["case_num"],
        )
        ds[field_name] = data_array
    return ds

In [15]:
emu_tracks_ec_earth3_veg_lr_historical_corrected = xr.open_dataset('../../03_NC_Wave_Emulator/03A_Stochastic_GCMs_NC/outputs/tc_logit/ec_earth3_veg_lr/historical/emu_tracks_ec_earth3_veg_lr_historical_corrected.nc')

emu_tracks_ec_earth3_veg_lr_historical_corrected

<xarray.Dataset> Size: 73MB
Dimensions:            (storm_number: 2455, time: 361, basin: 7, month: 12)
Coordinates:
  * time               (time) float64 3kB 0.0 3.6e+03 ... 1.292e+06 1.296e+06
  * basin              (basin) <U2 56B 'AU' 'EP' 'NA' 'NI' 'SI' 'SP' 'WP'
  * month              (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
    year               (storm_number) int64 20kB ...
    storm              (storm_number) int64 20kB ...
Dimensions without coordinates: storm_number
Data variables: (12/18)
    lon_trks           (storm_number, time) float64 7MB ...
    lat_trks           (storm_number, time) float64 7MB ...
    u250_trks          (storm_number, time) float64 7MB ...
    v250_trks          (storm_number, time) float64 7MB ...
    u850_trks          (storm_number, time) float64 7MB ...
    v850_trks          (storm_number, time) float64 7MB ...
    ...                 ...
    tc_days            (storm_number) float64 20kB ...
    seeds_per_month    (storm_number, basin, month) float64 2MB ...
    original_storm_id  (storm_number) int64 20kB ...
    time_emulation     (storm_number) datetime64[ns] 20kB ...
    sim_number         (storm_number) int64 20kB ...
    pmin_trks          (storm_number, time) float64 7MB ...

In [17]:
emu_tracks_ec_earth3_veg_lr_historical_corrected = xr.open_dataset('../../03_NC_Wave_Emulator/03A_Stochastic_GCMs_NC/outputs/tc_logit/ec_earth3_veg_lr/historical/emu_tracks_ec_earth3_veg_lr_historical_corrected.nc')
emu_tracks_ec_earth3_veg_lr_historical_corrected_out = "outputs/emu_tracks_ec_earth3_veg_lr_historical_corrected_param.csv"
extract_synthetic_track_params(emu_tracks_ec_earth3_veg_lr_historical_corrected, 
                               lat_site, 
                               lon_site, 
                               RADIUS_DEG, 
                               emu_tracks_ec_earth3_veg_lr_historical_corrected_out, 
                               fig_format='figures/13_syntetic_params_new/emu_tracks_ec_earth3_veg_lr_historical_corrected/storm_{i}_comparison.png')
# reconstruct_synthetic_tracks(emu_tracks_ec_earth3_veg_lr_historical_corrected_out, 
#                              "outputs/predicted_emu_tracks_ec_earth3_veg_lr_historical_corrected.nc", 
#                              "figures/13_syntetic_params_new/emu_tracks_ec_earth3_veg_lr_historical_corrected/reconstruction_case_{i}.png")

Extracted parameters for 907 tracks.
Tracks with too few points:: 61
Tracks outside 1487


In [18]:
emu_tracks_ec_earth3_veg_lr_ssp245_corrected = xr.open_dataset("../../03_NC_Wave_Emulator/03A_Stochastic_GCMs_NC/outputs/tc_logit/ec_earth3_veg_lr/ssp245/emu_tracks_ec_earth3_veg_lr_ssp245_corrected.nc")
emu_tracks_ec_earth3_veg_lr_ssp245_corrected_out = "outputs/emu_tracks_ec_earth3_veg_lr_ssp245_corrected_param.csv"
extract_synthetic_track_params(emu_tracks_ec_earth3_veg_lr_ssp245_corrected, 
                               lat_site, lon_site, 
                               RADIUS_DEG, 
                               emu_tracks_ec_earth3_veg_lr_ssp245_corrected_out, 
                               fig_format='figures/13_syntetic_params_new/emu_tracks_ec_earth3_veg_lr_ssp245_corrected/storm_{i}_comparison.png')
# reconstruct_synthetic_tracks(emu_tracks_ec_earth3_veg_lr_ssp245_corrected_out, 
#                              "outputs/predicted_emu_tracks_ec_earth3_veg_lr_ssp245_corrected.nc", 
#                              "figures/13_syntetic_params_new/emu_tracks_ec_earth3_veg_lr_ssp245_corrected/reconstruction_case_{i}.png")

Extracted parameters for 1828 tracks.
Tracks with too few points:: 144
Tracks outside 3267


In [19]:
emu_tracks_ec_earth3_veg_lr_ssp585_corrected = xr.open_dataset("../../03_NC_Wave_Emulator/03A_Stochastic_GCMs_NC/outputs/tc_logit/ec_earth3_veg_lr/ssp585/emu_tracks_ec_earth3_veg_lr_ssp585_corrected.nc")
emu_tracks_ec_earth3_veg_lr_ssp585_corrected_out = "outputs/emu_tracks_ec_earth3_veg_lr_ssp585_corrected_param.csv"
extract_synthetic_track_params(emu_tracks_ec_earth3_veg_lr_ssp585_corrected, 
                               lat_site, 
                               lon_site, 
                               RADIUS_DEG, 
                               emu_tracks_ec_earth3_veg_lr_ssp585_corrected_out, 
                               fig_format='figures/13_syntetic_params_new/emu_tracks_ec_earth3_veg_lr_ssp585_corrected/storm_{i}_comparison.png')
# reconstruct_synthetic_tracks(emu_tracks_ec_earth3_veg_lr_ssp585_corrected_out, 
#                              "outputs/predicted_emu_tracks_ec_earth3_veg_lr_ssp585_corrected.nc", 
#                              "figures/13_syntetic_params_new/emu_tracks_ec_earth3_veg_lr_ssp585_corrected/reconstruction_case_{i}.png")

Extracted parameters for 1785 tracks.
Tracks with too few points:: 112
Tracks outside 2819


In [ ]:
fealds = ['original_storm_id', 'time_emulation', 'sim_number']

In [ ]:
predicted_emu_tracks_ec_earth3_veg_lr_historical_corrected = xr.open_dataset("outputs/predicted_emu_tracks_ec_earth3_veg_lr_historical_corrected.nc")

predicted_emu_tracks_ec_earth3_veg_lr_historical_corrected_2 = add_field_2_dataset(
    predicted_emu_tracks_ec_earth3_veg_lr_historical_corrected, 
    emu_tracks_ec_earth3_veg_lr_historical_corrected_out, 
    fealds)

predicted_emu_tracks_ec_earth3_veg_lr_historical_corrected_2.to_netcdf("outputs/predicted_emu_tracks_ec_earth3_veg_lr_historical_corrected_mf.nc")

In [27]:
predicted_emu_tracks_ec_earth3_veg_lr_ssp245_corrected = xr.open_dataset("outputs/predicted_emu_tracks_ec_earth3_veg_lr_ssp245_corrected.nc")

predicted_emu_tracks_ec_earth3_veg_lr_ssp245_corrected_2 = add_field_2_dataset(
    predicted_emu_tracks_ec_earth3_veg_lr_ssp245_corrected, 
    emu_tracks_ec_earth3_veg_lr_ssp245_corrected_out, 
    fealds)

predicted_emu_tracks_ec_earth3_veg_lr_ssp245_corrected_2.to_netcdf("outputs/predicted_emu_tracks_ec_earth3_veg_lr_ssp245_corrected_mf.nc")

In [29]:
predicted_emu_tracks_ec_earth3_veg_lr_ssp585_corrected = xr.open_dataset("outputs/predicted_emu_tracks_ec_earth3_veg_lr_ssp585_corrected.nc")

predicted_emu_tracks_ec_earth3_veg_lr_ssp585_corrected_2 = add_field_2_dataset(
    predicted_emu_tracks_ec_earth3_veg_lr_ssp585_corrected, 
    emu_tracks_ec_earth3_veg_lr_ssp585_corrected_out,
    fealds)

predicted_emu_tracks_ec_earth3_veg_lr_ssp585_corrected_2.to_netcdf("outputs/predicted_emu_tracks_ec_earth3_veg_lr_ssp585_corrected_mf.nc")